In [55]:
# IMPORT LIBRARIES

In [56]:
import pandas as pd
import sqlite3

In [57]:
# CONNECT DATABASE

In [58]:
conn = sqlite3.connect('ipl_data.db')
print("Database connected successfully")

Database connected successfully


In [59]:
# CTE — SEASON-BY-SEASON WIN% (REWRITTEN)

In [60]:
Query = """
    WITH season_stats AS(
    SELECT season,
           COUNT(*) as total_matches,
           COUNT(CASE WHEN winner LIKE 'Royal Challengers%' THEN 1 END) as wins
    FROM matches
    where team1 LIKE 'Royal Challengers%' or team2 LIKE 'Royal Challengers%'
    GROUP BY season
    )
    SELECT season,
           total_matches
           wins,
           ROUND(wins*100/total_matches,1) as win_pct
           
    FROM season_stats
    ORDER BY season
    """
season_win_pct  = pd.read_sql_query(Query,conn)
print("Season win% (via CTE):")
print(season_win_pct)

Season win% (via CTE):
     season  wins  win_pct
0   2007/08    14     28.0
1      2009    16     56.0
2   2009/10    16     50.0
3      2011    16     62.0
4      2012    15     53.0
5      2013    16     56.0
6      2014    14     35.0
7      2015    16     50.0
8      2016    16     56.0
9      2017    13     23.0
10     2018    14     42.0
11     2019    14     35.0
12  2020/21    15     46.0
13     2021    15     60.0
14     2022    16     56.0
15     2023    14     50.0
16     2024    15     46.0


In [61]:
# CTE — TOSS IMPACT ANALYSIS (REWRITTEN)

In [62]:
Query = """WITH toss_impact AS(
    SELECT 
           COUNT(*) as total_matches,
           CASE WHEN toss_winner LIKE 'Royal Challengers%' THEN 'won_toss' ELSE 'lost_toss' END as toss_result,
           COUNT(CASE WHEN winner LIKE 'Royal Challengers%' THEN 1 END) as wins
    FROM matches
    where team1 LIKE 'Royal Challengers%' or team2 LIKE 'Royal Challengers%'
    GROUP BY toss_result
    )
    SELECT toss_result,
           total_matches,
           wins,
           ROUND(wins*100.0/total_matches,1) as win_pct
           
    FROM toss_impact
    """
toss_analysis = pd.read_sql_query(Query, conn)
print("Toss analysis (via CTE):")
print(toss_analysis)

Toss analysis (via CTE):
  toss_result  total_matches  wins  win_pct
0   lost_toss            134    62     46.3
1    won_toss            121    61     50.4


In [63]:
# CTE — ABOVE-AVERAGE POTM (REWRITTEN)

In [64]:
Query = """
    WITH potm_counts AS(
    SELECT 
           player_of_match,
           COUNT(*) as potm_count
    FROM matches
    where winner LIKE 'Royal Challengers%'
    GROUP BY player_of_match
    ORDER BY potm_count DESC
    ),
    avg_potm_count AS(
        SELECT SUM(potm_count)*1.0/ COUNT(player_of_match) as avg_count
        FROM potm_counts
    )
    SELECT  potm_counts.player_of_match,
            potm_counts.potm_count
    FROM potm_counts
    JOIN avg_potm_count
    where potm_counts.potm_count > avg_potm_count.avg_count
    """
above_avg_potm = pd.read_sql_query(Query, conn)
print("Players with above-average POTM awards (via CTE):")
print(above_avg_potm)

Players with above-average POTM awards (via CTE):
  player_of_match  potm_count
0  AB de Villiers          23
1         V Kohli          17
2        CH Gayle          17
3       JH Kallis           5
4       YS Chahal           4
5      GJ Maxwell           4
6    F du Plessis           4


In [65]:
# CTE — TOP RCB BATSMAN PER SEASON

In [66]:
query = """
WITH season_batting AS (
    SELECT 
    m.season,
    d.batter,
    SUM(d.batsman_runs) AS total_runs
    FROM deliveries d
    JOIN matches m ON d.match_id = m.id
    WHERE (m.team1 LIKE 'Royal Challengers%' OR m.team2 LIKE 'Royal Challengers%')
    AND d.batting_team LIKE 'Royal Challengers%'
    GROUP BY m.season, d.batter
),
ranked AS (
    SELECT 
    season,
    batter,
    total_runs,
    RANK() OVER (PARTITION BY season ORDER BY total_runs DESC) AS season_rank
    FROM season_batting
)
SELECT 
    season,
    batter,
    total_runs
FROM ranked
WHERE season_rank = 1
ORDER BY season
"""

top_per_season = pd.read_sql_query(query, conn)
print("Top RCB batsman per season (via CTE):")
print(top_per_season)

Top RCB batsman per season (via CTE):
     season          batter  total_runs
0   2007/08        R Dravid         371
1      2009       JH Kallis         361
2   2009/10       JH Kallis         572
3      2011        CH Gayle         608
4      2012        CH Gayle         733
5      2013        CH Gayle         720
6      2014  AB de Villiers         395
7      2015  AB de Villiers         513
8      2016         V Kohli         973
9      2017         V Kohli         308
10     2018         V Kohli         530
11     2019         V Kohli         464
12  2020/21      D Padikkal         473
13     2021      GJ Maxwell         513
14     2022    F du Plessis         468
15     2023    F du Plessis         730
16     2024         V Kohli         741


In [67]:
# CTE — MATCHES WHERE RCB SCORED 180+ AND WON

In [68]:
query = """
WITH rcb_scores AS (
    SELECT 
    match_id,
    SUM(total_runs) AS team_score
    FROM deliveries
    WHERE batting_team LIKE 'Royal Challengers%'
    GROUP BY match_id
),
rcb_wins AS (
    SELECT 
    id AS match_id,
    season,
    CASE 
    WHEN team1 LIKE 'Royal Challengers%' THEN team2
    ELSE team1
    END AS opponent
    FROM matches
    WHERE winner LIKE 'Royal Challengers%'
)
SELECT 
    s.match_id,
    w.season,
    s.team_score,
    w.opponent
    FROM rcb_scores s
    JOIN rcb_wins w ON s.match_id = w.match_id
    WHERE s.team_score >= 180
    ORDER BY s.team_score DESC
"""

high_score_wins = pd.read_sql_query(query, conn)
print("Matches where RCB scored 180+ and won:")
print(high_score_wins.head(10))
print(f"\nTotal such matches: {len(high_score_wins)}")

Matches where RCB scored 180+ and won:
   match_id season  team_score             opponent
0    598027   2013         263        Pune Warriors
1    980987   2016         248        Gujarat Lions
2   1426296   2024         241         Punjab Kings
3    829795   2015         235       Mumbai Indians
4    980907   2016         227  Sunrisers Hyderabad
5    829785   2015         226      Kings XI Punjab
6   1136611   2018         218  Sunrisers Hyderabad
7   1426306   2024         218  Chennai Super Kings
8    548372   2012         215     Delhi Daredevils
9   1082610   2017         213        Gujarat Lions

Total such matches: 42


In [69]:
# CLOSE CONNECTION

In [54]:
conn.close()
print("Connection closed.")

Connection closed.


## Day 11 Summary: CTEs (Common Table Expressions)

 Completed
- Rewrote season win% query using CTE — same result, cleaner code
- Rewrote toss analysis using CTE
- Rewrote above-average POTM using two CTEs (potm_counts + avg_potm)
- Found top batsman per season using CTE + RANK window function
- Identified matches where RCB scored 180+ and won (using two CTEs + JOIN)

 Key Concepts Learned
- WITH cte_name AS ( ... ) — defines a named temporary result set
- Multiple CTEs separated by commas: WITH cte1 AS (...), cte2 AS (...)
- CTE + Window Functions (RANK with PARTITION BY)
- CTEs replacing nested subqueries for readability
- Each CTE is independently testable

 Why This Matters
- Industry-standard way to write complex queries
- Easier to debug — test each CTE separately
- More readable than deeply nested subqueries
- Common in data engineering and analytics roles

 Next
Day 12 — Views: Creating permanent reusable queries in the database